# StructConformity

Classificação de conformidade estrutural segundo a NBR 6118 via Machine Learning.

**Autor:** Renan Araújo  
**Disciplina:** Pós-graduação em Engenharia de Software — PUC-RJ  
**Dataset:** Sintético, gerado a partir das regras da NBR 6118 (vigas e pilares)  
**Algoritmos testados:** KNN, Árvore de Decisão, Naive Bayes, SVM


## Sobre este notebook

O notebook cobre o pipeline de ML do projeto:

1. Carga e exploração do dataset
2. Pré-processamento (encoding, holdout, Pipelines com scaler)
3. Treinamento dos quatro algoritmos
4. Otimização de hiperparâmetros via GridSearchCV com cross-validation
5. Avaliação comparativa
6. Exportação do modelo para o backend
7. Reflexão sobre segurança e anonimização


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from sklearn.metrics import precision_score, recall_score, f1_score


## 1. Carga do Dataset

O dataset é sintético, gerado em `dataset/generate_dataset.py` a partir das regras da NBR 6118. São 20.000 registros de elementos estruturais (vigas e pilares), balanceados 50% conformes / 50% não conformes. Cada registro tem 10 features numéricas/categóricas e a classificação binária.


## Regras de conformidade implementadas

Cada registro é classificado como **conforme** ou **não conforme** a partir de 12 regras (10 reprovam, 2 são alertas). As regras foram extraídas da NBR 6118 e de boas práticas construtivas.

| # | Regra | Aplica a | Motivo de reprovação |
|---|-------|----------|----------------------|
| 1 | Cobrimento ≥ 2,5 cm | ambos | `cobrimento_insuficiente` |
| 2 | fck ≥ 20 MPa | ambos | `fck_invalido` |
| 3 | Largura mínima: viga ≥ 12 cm, pilar ≥ 19 cm | ambos | `geometria_invalida` |
| 4 | Quantidade mínima de barras longitudinais ≥ 4 | ambos | `quantidade_insuficiente` |
| 5 | Bitola do estribo ≥ 5,0 mm | ambos | `stirrup_diam_invalido` |
| 6 | Bitola longitudinal: viga ≥ 8 mm, pilar ≥ 10 mm | ambos | `main_rebar_diam_invalido` |
| 7 | Espaçamento máximo de estribos: viga `min(0,6·dim_b; 30)`, pilar `min(20; menor_dim; 12·φ_long)` | ambos | `stirrup_spacing_excedido` |
| 8 | ρ mínimo — viga: tabela NBR por fck (0,15%–0,208%); pilar: 0,4% | ambos | `rho_insuficiente` |
| 9 | ρ máximo — viga: 4%; pilar: 8% | ambos | `rho_excedido` |
| 10 | Espaçamento geométrico: as barras cabem na seção respeitando `s_min = max(φ; 2,0; 1,2·φ_agregado)` | ambos | `espacamento_insuficiente` |

**Alertas (não reprovam):**

- Espaçamento de estribos múltiplo de 5 cm — boa prática construtiva.

### Observações de domínio

A taxa de armadura ρ = As/Ac é a regra mais rica pedagogicamente porque combina quatro features (`main_rebar_diam`, `main_rebar_quantity`, `dim_a`, `dim_b`) em uma razão. A Regra 10 valida fisicamente se as barras cabem: para vigas, testa de 1 a 4 camadas; para pilares, distribui em grade perimetral de 4 faces e verifica o espaçamento em cada direção. Isso elimina combinações fisicamente impossíveis (muitas barras em seções pequenas).


In [ ]:
url = "https://raw.githubusercontent.com/Renanarauujo/StructConformity/refs/heads/master/dataset/structural_conformity.csv"
df = pd.read_csv(url)
df.head(10)

In [ ]:
print(f"Linhas: {df.shape[0]}, Colunas: {df.shape[1]}")
print()
df.info()

In [ ]:
df.describe()

In [ ]:
print(df["conformity"].value_counts())
print()
print(df["conformity"].value_counts(normalize=True).round(2))

### 1.1 Visualizacoes

 Gráficos para entender a distribuição dos dados e a relação entre as features.

In [ ]:
# Distribuição das classes (gráfico de barras)
fig, ax = plt.subplots(figsize=(6, 4))
df["conformity"].value_counts().plot(kind="bar", color=["#2980b9", "#1a5276"], ax=ax)
ax.set_title("Distribuição de classes")
ax.set_ylabel("Quantidade")
ax.set_xlabel("Classe")
ax.tick_params(axis="x", rotation=0)
ax.set_xticklabels(["Conforme", "Não conforme"])
plt.tight_layout()
plt.show()

In [ ]:
# Histogramas das features numéricas
numeric_cols = df.select_dtypes(include=[np.number]).columns

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=20, edgecolor="black", alpha=0.7)
    axes[i].set_title(col)

plt.suptitle("Distribuição das Features Numéricas", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlação
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Matriz de Correlação entre Features")
plt.tight_layout()
plt.show()

## 2. Pré-processamento

  Preparação dos dados para treinamento dos modelos:
  1. Codificação da variável categórica (element_type)
  2. Separação de features (X) e target (y)
  3. Divisão treino/teste (holdout 70/30)
  4. Normalização e padronização via Pipeline (para evitar Data Leakage)

In [ ]:
# Codificação do element_type
le = LabelEncoder()
df["element_type_encoded"] = le.fit_transform(df["element_type"])

print("Mapeamento:")
for i, classe in enumerate(le.classes_):
    print(f"  {classe} -> {i}")

In [ ]:
X = df[["element_type_encoded", "dim_a", "dim_b", "dim_c", "fck", "cover",
        "main_rebar_diam", "main_rebar_quantity", "stirrup_diam", "stirrup_spacing"]]
y = df["conformity"].map({"conforme": 1, "nao_conforme": 0})

print("Shape X:", X.shape)
print("Shape y:", y.shape)
print("Distribuicao y:", y.value_counts().to_dict())


In [ ]:
# Dividir treino/teste
X_train, X_test, y_train, y_test = train_test_split(
      X, y, test_size=0.3, random_state=42, stratify=y
  )

print(f"Treino: {X_train.shape[0]} amostras ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Teste:  {X_test.shape[0]} amostras ({X_test.shape[0]/len(X)*100:.0f}%)")

 ## 3. Modelagem

  Treinamento de 4 algoritmos de classificação, cada um encapsulado em um Pipeline (scaler + modelo) para evitar Data Leakage.

  **Algoritmos:**
  1. KNN (K-Nearest Neighbors)
  2. Arvore de Classificação (Decision Tree)
  3. Naive Bayes (GaussianNB)
  4. SVM (Support Vector Machine)

In [ ]:
pipelines = {
      "KNN": Pipeline([
          ("scaler", StandardScaler()),
          ("classifier", KNeighborsClassifier())
      ]),
      "Decision Tree": Pipeline([
          ("scaler", StandardScaler()),
          ("classifier", DecisionTreeClassifier(random_state=42))
      ]),
      "Naive Bayes": Pipeline([
          ("scaler", StandardScaler()),
          ("classifier", GaussianNB())
      ]),
      "SVM": Pipeline([
          ("scaler", StandardScaler()),
          ("classifier", SVC(random_state=42))
      ]),
  }

results = {}

for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name:15s} -> Acuracia: {acc:.4f} ({acc*100:.1f}%)")

 ## 4. Otimização de Hiperparâmetros

  Busca (GridSearchCV) pela melhor combinação de hiperparâmetros de cada algoritmo, usando cross-validation com 5 folds.

In [ ]:
param_grids = {
      "KNN": {
          "classifier__n_neighbors": [3, 5, 7, 9, 11],
          "classifier__metric": ["euclidean", "manhattan"],
          "classifier__weights": ["uniform", "distance"],
      },
      "Decision Tree": {
          "classifier__max_depth": [3, 5, 10, 15, None],
          "classifier__criterion": ["gini", "entropy"],
          "classifier__min_samples_split": [2, 5, 10],
      },
      "Naive Bayes": {
          "classifier__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-5],
      },
      "SVM": {
          "classifier__C": [0.1, 1, 10, 100],
          "classifier__kernel": ["linear", "rbf"],
          "classifier__gamma": ["scale", "auto"],
      },
  }

In [ ]:
best_pipelines = {}

for name, pipeline in pipelines.items():
    print(f"Otimizando {name}...")

    grid = GridSearchCV(
        pipeline,
        param_grids[name],
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    best_pipelines[name] = grid.best_estimator_

    print(f"  Melhores parametros: {grid.best_params_}")
    print(f"  Melhor acuracia (CV): {grid.best_score_:.4f}")
    print()

In [ ]:
print("Comparação: modelos otimizados no conjunto de teste\n")

optimized_results = {}

for name, model in best_pipelines.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    optimized_results[name] = acc
    print(f"{name:15s} -> Antes: {results[name]:.4f} | Depois: {acc:.4f} | Diferença: {acc - results[name]:+.4f}")

## 5. Avaliacao dos Modelos

  Avaliação detalhada dos modelos otimizados com matriz de confusão e métricas por classe.

  **Metricas:**
  - **Acurácia:** taxa de acerto geral
  - **Precisão:** dos que disse "conforme", quantos realmente sao
  - **Recall:** dos que sao "nao conforme", quantos identificou
  - **F1-Score:** media harmonica entre precisao e recall

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (name, model) in enumerate(best_pipelines.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["NC", "C"])
    disp.plot(ax=axes[i], cmap="Blues", colorbar=False)
    axes[i].set_title(f"{name} (Acc: {optimized_results[name]:.1%})")

plt.suptitle("Matrizes de Confusão - Modelos Otimizados", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
for name, model in best_pipelines.items():
      y_pred = model.predict(X_test)
      print(f"{'='*50}")
      print(f" {name}")
      print(f"{'='*50}")
      print(classification_report(y_test, y_pred, target_names=["Nao Conforme", "Conforme"]))

In [ ]:
comparison = []

for name, model in best_pipelines.items():
    y_pred = model.predict(X_test)
    comparison.append({
        "Modelo": name,
        "Acurácia": accuracy_score(y_test, y_pred),
        "Precisão": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Recall Não Conforme": recall_score(y_test, y_pred, pos_label=0),
      })

df_comparison = pd.DataFrame(comparison).set_index("Modelo")
df_comparison.style.format("{:.4f}").highlight_max(axis=0, color="#c6efce")

### 5.1 Análise final

A **Árvore de Decisão** lidera todas as métricas por uma razão estrutural do problema: o dataset foi gerado por 10 regras determinísticas (cobrimento ≥ 2,5, fck ≥ 20, ρ ≤ limite por tipo, etc.). Uma árvore aprende exatamente esse formato,  `feature ≥ threshold`, então a fronteira de decisão do modelo praticamente reproduz a norma.

KNN, Naive Bayes e SVM usam distância, probabilidade e hiperplano separador; nenhuma delas se ajusta tão bem a regras em cascata com thresholds duros.

**Os erros remanescentes** ocorrem em casos borderline de ρ (taxa próxima do limite por poucos décimos) ou em combinações raras de `(qty, φ, dim_a, dim_b)`. Como a árvore aprende por thresholds em uma feature por vez, ela aproxima mal razões multiplicativas como ρ = π·φ²·qty / (4·dim_a·dim_b). Em produção, isso seria endereçado com um guard-rail determinístico que aplica as regras diretamente antes do ML.

**Modelo exportado:** Árvore de Decisão, por combinar melhor acurácia global e maior recall para a classe *não conforme*, a métrica mais crítica aqui, já que aprovar uma peça defeituosa tem consequência real.


In [ ]:
# Importação do modelo
import joblib

best_model = best_pipelines["Decision Tree"]
joblib.dump(best_model, "../backend/model/best_model.pkl")

print("Modelo exportado: backend/model/best_model.pkl")
print(f"Tipo: {type(best_model)}")
print(f"Pipeline: {best_model.steps}")